# Use pymilvus query iterator to pull from database into dataframe

In [84]:
from pymilvus import MilvusClient
from dotenv import load_dotenv
import os
import pandas as pd
import re

load_dotenv(override=True)

True

In [ ]:
collection_name = "pptx_fixed250_tag30_prompt7"

In [85]:
fname_postfix = re.sub(r"pptx_", "", collection_name)
fname_postfix = re.sub(r"_tag_v\d+", "", fname_postfix)
fname_postfix

'fixed_500words'

In [56]:
client = MilvusClient(
    uri = os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token = os.getenv("ZILLIZ_CLUSTER_TOKEN"),
)

In [57]:
client.load_collection(collection_name=collection_name)

In [ ]:
iterator = client.query_iterator(
    batch_size=10,
    collection_name=collection_name,
    output_fields=["*"],
)

2025-03-17 08:37:43,627 [WARNING][__setup_ts_by_request]: failed to get mvccTs from milvus server, use client-side ts instead (iterator.py:260)


In [ ]:
res = iterator.next()
res

In [ ]:
for batch in iterator:
    "Do something"

# Label by document name

In [88]:
import glob
from pymilvus import MilvusClient
from dotenv import load_dotenv
import os
import pandas as pd
import pickle
import re

load_dotenv(override=True)

files = glob.glob("**/*.pptx", recursive=True)
files

['Content/AWS introduction/AWS VPC & Networking.pptx',
 'Content/AWS introduction/RDS-Lecture.pptx',
 'Content/AWS introduction/Introduction to Cloud.pptx',
 'Content/AWS introduction/Introduction to S3.pptx',
 'Content/AWS introduction/Introduction to EC2.pptx',
 'Content/AWS introduction/Introduction to AWS.pptx',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Lecture_LLM_Data_Preparation.pptx',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/~$Lecture_LLM_Data_Preparation.pptx',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Lecture_LLM_Text_Data_Augmentation.pptx',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Collection/~$Lecture_LLM_Data_Collection.pptx',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Collection/Lecture_LLM_Data_Collection.pptx',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Annotation/Lecture_LLM_Data_Lab

In [101]:
collection_name = "pptx_fixed250_tag30_prompt7"

client = MilvusClient(
    uri = os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token = os.getenv("ZILLIZ_CLUSTER_TOKEN"),
)
client.load_collection(collection_name)

In [102]:
fname_postfix = re.sub("pptx_", "", collection_name)
fname_postfix = re.sub(r"_tag\S+", "", fname_postfix)
fname_postfix

'fixed250'

In [ ]:
os.makedirs("./labeling/pptx/", exist_ok=True)

d = {
    "id": [],
    "text": [],
    "filename": [],
    "label": [],
}
for file in files:
    head, tail = os.path.split(file)
    # res = client.query(
    #     collection_name=collection_name,
    #     filter=f'metadata["title"] == "{tail}"',
    #     output_fields=["id", "text", "metadata"]
    # )
    expr = 'metadata["title"] == {tail}'
    filter_params = {"tail": tail}
    res = client.query(
        collection_name=collection_name,
        filter=expr,
        filter_params=filter_params,
        output_fields=["id", "text", "metadata"]
    )
    # for r in res:
    #     print(r["metadata"]["title"])

    if re.search("AWS", file):
        label = ["Infrastructure and Operations"]
    elif re.search("Lecture_LLM_Data_Preparation.pptx", file):
        # label = ['Machine Learning Engineering', 'Data Engineering']
        label = ["Data Engineering"]
    elif re.search("LLM_Data_Collection.pptx", file):
        # label = ['Data Science', 'Data Engineering']
        # label = ["Data Engineering"]
        label = ["Data Science"]
    elif re.search("LLM_Data_Labelling.pptx", file):
        label = ["Data Science"]
    elif re.search("Copy of 10.2 Search Engine.pptx", file):
        # label = ['Data Science', 'Machine Learning Engineering']
        label = ["Machine Learning Engineering"]
    elif re.search("Copy of m3.1-data-prep-timeseries-lecture.pptx", file):
        label = ["Data Science"]
    elif re.search("Large Language Model", file):
        label = ["Machine Learning Engineering"]
    elif re.search("Metabase", file):
        label = ["Data Analysis"]

    for r in res:
        d["id"].append(r["id"])
        d["text"].append(r["text"])
        d["filename"].append(file)
        d["label"].append(label)

    # for r in res:
    #     fname_write = f"./labeling/pptx/{tail}-{r["id"]}.txt"
    #     to_write = f'ID: {r["id"]}\nFile: {file}\nGround Truth Label: ["{label}"]\n\n{r["text"]}'
    #     with open(fname_write, "w") as f:
    #         f.write(to_write)

df = pd.DataFrame.from_dict(d)
df.to_pickle(f"./labeling/pptx/df_labels_{fname_postfix}.pkl")
df.to_csv(f"./labeling/pptx/df_labels_{fname_postfix}.csv")

In [95]:
df.filename.value_counts()

filename
Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Lecture_LLM_Data_Preparation.pptx                                                     24
Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Lecture_LLM_Text_Data_Augmentation.pptx                                            23
Content/Large Language Model ( LLM)/Part_9_RAG-20250225T182503Z-001/Part_9_RAG/Lecture_RAG_Introduction.pptx                                                       22
Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Annotation/Lecture_LLM_Data_Labelling.pptx                                                        22
Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Collection/Lecture_LLM_Data_Collection.pptx                                                       18
Content/Large Language Model ( LLM)/Part_10_LLM_Use_Cases-20250225T182513Z-001/Part_10_LLM_Use_Cases/Copy of 10.2 Search Engine.pptx                             

In [73]:
df.head()

,id,text,filename,label
0,1cee057ffe9578f22ff725be85f00caf53b0da8e275ca7...,Multi-AZ deployments for high availability and...,Content/AWS introduction/AWS VPC & Networking....,Infrastructure and Operations
1,6f751b1bc650e4b08f46f46dee65cbaf0d7f48f981f381...,AWS VPC & Networking References\nhttps://www.s...,Content/AWS introduction/AWS VPC & Networking....,Infrastructure and Operations
2,4a85ffea7658fab70f26160859ad8ebed8d2338dc90f48...,RDS\nReference: https://www.slideshare.net/sli...,Content/AWS introduction/RDS-Lecture.pptx,Infrastructure and Operations
3,ab26be99457f22c8c78d85b99f0e08854b476b8d083a93...,Introduction to Cloud Computing Prepared by We...,Content/AWS introduction/Introduction to Cloud...,Infrastructure and Operations
4,01e7692c1a39a9b9590247b79f7a27bfa5b00b0045124f...,Introduction to S3 Prepared by WeCloudData Age...,Content/AWS introduction/Introduction to S3.pptx,Infrastructure and Operations


In [ ]:
res = client.query(
    collection_name=collection_name,
    filter='metadata["title"] == "Lecture_LLM_Data_Preparation.pptx"',
    output_fields=["id", "text", "metadata"]
)

res

data: ["{'id': '0db7212702ce7c51e56ff63796a7a04b07e7cf2e0064cc3e0414abf1acf0d710', 'text': 'Additionally, since most of the downstream tasks they evaluated on focused on English-language text, they used langdetect to filter out any pages that were not classified as English with a probability of at least 0.99\\nPreparing the C4 Dataset - 2\\x0cAutomatic filtering method (using WebText as a proxy for high quality documents), to improve the quality of Common Crawl (using a logistic regression based classifier).\\nFuzzy de-duplication of documents using a hashing method (MiniHashLSH)\\nPartial removal of text occurring in benchmark datasets that appears in CommonCrawl/WebText as well. “Unfortunately, a bug resulted in only partial removal of all detected overlaps from the training data. Due to the cost of training, it wasn’t feasible to retrain the model.”\\nPreparing the data for GPT-3\\nBrown et.al. (2020)\\x0cUses jusText on Web Archive files (raw HTTP responses including page HTML) for

In [34]:
for r in res:
    print(r["metadata"]["title"])

Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
Lecture_LLM_Data_Preparation.pptx
